# Bike Station Graph Preparation

Prepare the Toronto bike-share data for analysis on the project city/network graph.

This notebook keeps the scope to the shared city graph:

- load Toronto `locations_gdf` with census tract geometry
- load/build `network_nodes` and `network_edges`
- spatially map bike stations to graph `loc_id`
- annotate bike trips with start/end graph locations
- store the PyG `network_graph` object for later modelling

In [1]:
import polars as pl
from pathlib import Path
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import Point
from scipy.optimize import minimize

In [2]:
project_root = Path("/home/najla/dev/najla-msc/bikeshare/")

In [ ]:
"""
dfs neeeded:-
df_baseline: used for labeled data, drop node2vect dim and text features
node_df: merge with season and temporal features, merge with wikidata features
"""

In [3]:
"""locations_path = Path(
    "/home/najla/dev/najla-msc/data/processed/THATS/TorontoData/locations_gdf.parquet"
)
"""
locations_path = Path(
    "/home/najla/dev/najla-msc/bikeshare/data/processed/graph/locations_gdf.parquet"
)

locations_gdf = gpd.read_parquet(locations_path,
    columns=["loc_id", "loc_name","geometry"],)

locations_gdf.head()

,loc_id,loc_name,geometry
859,5320100.01,0100.01,"POLYGON ((-78.88335 43.85639, -78.88219 43.852..."
860,5320100.02,0100.02,"POLYGON ((-78.98162 43.86302, -78.98308 43.862..."
861,5320100.03,0100.03,"POLYGON ((-78.95885 43.89375, -78.95846 43.892..."
527,5320105.14,0105.14,"POLYGON ((-78.96559 43.93508, -78.96519 43.934..."
528,5320105.17,0105.17,"POLYGON ((-79.0165 43.94033, -79.01692 43.9402..."


In [4]:
# check the crs of the locations_gdf, geom name and type
print(locations_gdf.crs)
print(locations_gdf.geometry.name)
print(locations_gdf.geometry.type.count())

{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "scope": "Horizontal

In [5]:
"""
bike_trips_path = Path(
    "/home/najla/dev/najla-msc/bikeshare/data/processed/bike_ridership/bike_trips_stations_info_5yrs_12pm.parquet"
)"""
bike_trips_path = Path(
    "/home/najla/dev/najla-msc/bikeshare/data/processed/bike_ridership/bike_trips_stations_info_5yrs_12pm.parquet"
)

bike_trips = pl.read_parquet(bike_trips_path)

bike_trips.head()

end_date,end_station_name,start_station_name,avg_trip_duration,trip_count,end_station_id,lat_end,lon_end,capacity_end,start_station_id,lat_start,lon_start,capacity_start
date,str,str,f64,u32,i64,f64,f64,i64,i64,f64,f64,i64
2024-09-25,"""Central Tech (Harbord St)""","""Salem Ave / Bloor St W""",612.0,1,7191,43.661975,-79.407896,10,7156,43.660833,-79.431667,15
2024-09-25,"""Huron St / Harbord St""","""Bathurst Subway Station""",440.0,1,7058,43.6637,-79.400053,39,7154,43.666667,-79.411667,23
2024-09-26,"""Beverley St / College St""","""51 Parliament St""",759.0,1,7161,43.6575,-79.395278,23,7064,43.652169,-79.362841,19
2022-11-29,"""Yonge St / Alexander St - SMAR…","""Seaton St / Dundas St E - SMAR…",464.0,1,7271,43.662862,-79.383572,18,7109,43.658777,-79.369596,32
2023-07-31,"""Seaton St / Dundas St E - SMAR…","""Seaton St / Dundas St E - SMAR…",67.0,1,7109,43.658777,-79.369596,32,7109,43.658777,-79.369596,32


In [6]:
from datetime import date

bike_trips.filter(
    pl.col("end_date") > date(2025, 1, 1)
)

end_date,end_station_name,start_station_name,avg_trip_duration,trip_count,end_station_id,lat_end,lon_end,capacity_end,start_station_id,lat_start,lon_start,capacity_start
date,str,str,f64,u32,i64,f64,f64,i64,i64,f64,f64,i64
2025-11-06,"""Beverley St / College St""","""Widmer St / King St W""",351.0,1,7161,43.6575,-79.395278,23,7721,43.646357,-79.391206,11
2025-11-07,"""Union Station (North)""","""Bathurst St / Dundas St W""",944.0,1,8001,43.64565,-79.381163,24,7037,43.652208,-79.405569,41
2025-10-25,"""Portland St / Front St W""","""Portland St / Front St W""",825.4,5,8168,43.641634,-79.398789,11,8168,43.641634,-79.398789,11
2025-01-25,"""Seaton St / Dundas St E - SMAR…","""Seaton St / Dundas St E - SMAR…",366.4,5,7109,43.658777,-79.369596,32,7109,43.658777,-79.369596,32
2025-11-03,"""Central Tech (Harbord St)""","""Bathurst St / Dundas St W""",463.0,1,7191,43.661975,-79.407896,10,7037,43.652208,-79.405569,41
…,…,…,…,…,…,…,…,…,…,…,…,…
2025-12-17,"""East Liberty St / Western Batt…","""Bathurst St / Adelaide St W""",333.0,1,7322,43.639278,-79.411574,19,7298,43.645324,-79.40345,25
2025-12-18,"""Queen St W / John St""","""Gould St / Mutual St""",480.0,1,7542,43.650077,-79.391291,23,7028,43.6582,-79.3768,31
2025-12-18,"""Ulster St / Bathurst St""","""St. Andrew's Playground Park""",704.0,1,7195,43.66,-79.408889,10,7718,43.646552,-79.399586,11


In [7]:
start_stations = bike_trips.select(
    [
        "start_station_id",
        "start_station_name",
        "lon_start",
        "lat_start",
    ]
).rename(
    {
        "start_station_id": "station_id",
        "start_station_name": "station_name",
        "lon_start": "lon",
        "lat_start": "lat",
    }
)

end_stations = bike_trips.select(
    [
        "end_station_id",
        "end_station_name",
        "lon_end",
        "lat_end",
    ]
).rename(
    {
        "end_station_id": "station_id",
        "end_station_name": "station_name",
        "lon_end": "lon",
        "lat_end": "lat",
    }
)

In [8]:
stations = (
    pl.concat([start_stations, end_stations])
    .drop_nulls(["lon", "lat"])
    .unique()
)
stations.sort("station_name").head(426)

station_id,station_name,lon,lat
i64,str,f64,f64
7657,"""1 Market St""",-79.370666,43.646993
8009,"""1 Shortt St""",-79.451901,43.69593
7708,"""101 Cedarvale Ave""",-79.311094,43.686868
7710,"""11 Spadina Rd""",-79.404137,43.667725
7268,"""111 Bond St (North of Dundas S…",-79.378497,43.656927
…,…,…,…
7775,"""Grenoble Dr / Vendome Pl""",-79.328114,43.715538
8084,"""Grenville St / Yonge St""",-79.38361,43.662029
7624,"""Guildwood GO Station (South)""",-79.197129,43.754978


In [9]:
# no duplications in names, station_id, "lon","lat"
stations.group_by("lon","lat").len().filter(pl.col("len") > 1).sort("len", descending=True)

lon,lat,len
f64,f64,u32


In [10]:
stations_pd = stations.to_pandas()

stations_gdf = gpd.GeoDataFrame(
    stations_pd,
    geometry=gpd.points_from_xy(stations_pd["lon"], stations_pd["lat"]),
    crs="EPSG:4326",
)

In [11]:
stations_gdf.shape

(1007, 5)

In [12]:
station_locations = gpd.sjoin(
    stations_gdf,
    locations_gdf,
    how="left",
    predicate="within",
).drop(columns=["index_right"], errors="ignore")

station_locations

,station_id,station_name,lon,lat,geometry,loc_id,loc_name
0,7044,Church St / Alexander St,-79.380288,43.663722,POINT (-79.38029 43.66372),5350063.06,0063.06
1,7809,Firvalley Ct / Warden Ave,-79.278715,43.703211,POINT (-79.27871 43.70321),5350341.04,0341.04
2,7990,Don Mills Rd / Goodview Rd,-79.353682,43.785414,POINT (-79.35368 43.78541),5350304.01,0304.01
3,7925,Niagara St / King St W - SMART,-79.407514,43.642813,POINT (-79.40751 43.64281),5350010.02,0010.02
4,7123,420 Wellington St W,-79.396649,43.643834,POINT (-79.39665 43.64383),5350011.01,0011.01
...,...,...,...,...,...,...,...
1002,7523,Lynn Williams St / East Liberty St,-79.416866,43.638925,POINT (-79.41687 43.63892),5350008.01,0008.01
1003,7660,285 Victoria St,-79.379625,43.656633,POINT (-79.37962 43.65663),5350034.02,0034.02
1004,7794,York St / Harbour St (Love Park),-79.380359,43.641017,POINT (-79.38036 43.64102),5350013.02,0013.02
1005,8047,Lawrence Ave E / Overture Rd,-79.199615,43.765641,POINT (-79.19961 43.76564),5350359.00,0359.00


In [13]:
station_locations[station_locations["loc_id"].isna()].head()

,station_id,station_name,lon,lat,geometry,loc_id,loc_name


In [14]:
bike_trips_pd = bike_trips.to_pandas() if hasattr(bike_trips, "to_pandas") else bike_trips.copy()

bike_trips_pd.head()

,end_date,end_station_name,start_station_name,avg_trip_duration,trip_count,end_station_id,lat_end,lon_end,capacity_end,start_station_id,lat_start,lon_start,capacity_start
0,2024-09-25,Central Tech (Harbord St),Salem Ave / Bloor St W,612.0,1,7191,43.661975,-79.407896,10,7156,43.660833,-79.431667,15
1,2024-09-25,Huron St / Harbord St,Bathurst Subway Station,440.0,1,7058,43.663700,-79.400053,39,7154,43.666667,-79.411667,23
2,2024-09-26,Beverley St / College St,51 Parliament St,759.0,1,7161,43.657500,-79.395278,23,7064,43.652169,-79.362841,19
3,2022-11-29,Yonge St / Alexander St - SMART,Seaton St / Dundas St E - SMART,464.0,1,7271,43.662862,-79.383572,18,7109,43.658777,-79.369596,32
4,2023-07-31,Seaton St / Dundas St E - SMART,Seaton St / Dundas St E - SMART,67.0,1,7109,43.658777,-79.369596,32,7109,43.658777,-79.369596,32


In [15]:
bike_trips_locations =bike_trips_pd.merge(
    station_locations.rename(
        columns={
            "loc_id": "start_loc_id",
            "loc_name": "start_loc_name",
            "lon": "start_node_lon",
            "lat": "start_node_lat"
        }
    ),
    left_on="start_station_name",
    right_on="station_name",
    how="left",
).drop(
    columns=["station_name","station_id", "geometry"],
    errors="ignore",
).merge(
    station_locations.rename(
        columns={
            "loc_id": "end_loc_id",
            "loc_name": "end_loc_name",
            "lon": "end_node_lon",
            "lat": "end_node_lat"
        }
    ),
    left_on="end_station_name",
    right_on="station_name",
    how="left",
).drop(
    columns=["station_name","station_id", "geometry"],
    errors="ignore",
)
bike_trips_locations.head()

,end_date,end_station_name,start_station_name,avg_trip_duration,trip_count,end_station_id,lat_end,lon_end,capacity_end,start_station_id,...,lon_start,capacity_start,start_node_lon,start_node_lat,start_loc_id,start_loc_name,end_node_lon,end_node_lat,end_loc_id,end_loc_name
0,2024-09-25,Central Tech (Harbord St),Salem Ave / Bloor St W,612.0,1,7191,43.661975,-79.407896,10,7156,...,-79.431667,15,-79.431667,43.660833,5350096.02,0096.02,-79.407896,43.661975,5350060.00,0060.00
1,2024-09-25,Huron St / Harbord St,Bathurst Subway Station,440.0,1,7058,43.663700,-79.400053,39,7154,...,-79.411667,23,-79.411667,43.666667,5350092.00,0092.00,-79.400053,43.663700,5350061.00,0061.00
2,2024-09-26,Beverley St / College St,51 Parliament St,759.0,1,7161,43.657500,-79.395278,23,7064,...,-79.362841,19,-79.362841,43.652169,5350016.00,0016.00,-79.395278,43.657500,5350037.00,0037.00
3,2022-11-29,Yonge St / Alexander St - SMART,Seaton St / Dundas St E - SMART,464.0,1,7271,43.662862,-79.383572,18,7109,...,-79.369596,32,-79.369596,43.658777,5350032.00,0032.00,-79.383572,43.662862,5350063.06,0063.06
4,2023-07-31,Seaton St / Dundas St E - SMART,Seaton St / Dundas St E - SMART,67.0,1,7109,43.658777,-79.369596,32,7109,...,-79.369596,32,-79.369596,43.658777,5350032.00,0032.00,-79.369596,43.658777,5350032.00,0032.00


In [16]:
bike_trips_locations[bike_trips_locations["start_loc_id"] == bike_trips_locations["end_loc_id"]].count()

end_date              361132
end_station_name      361132
start_station_name    361132
avg_trip_duration     361132
trip_count            361132
end_station_id        361132
lat_end               361132
lon_end               361132
capacity_end          361132
start_station_id      361132
lat_start             361132
lon_start             361132
capacity_start        361132
start_node_lon        361132
start_node_lat        361132
start_loc_id          361132
start_loc_name        361132
end_node_lon          361132
end_node_lat          361132
end_loc_id            361132
end_loc_name          361132
dtype: int64

In [17]:
bike_trips_locations.columns

Index(['end_date', 'end_station_name', 'start_station_name',
       'avg_trip_duration', 'trip_count', 'end_station_id', 'lat_end',
       'lon_end', 'capacity_end', 'start_station_id', 'lat_start', 'lon_start',
       'capacity_start', 'start_node_lon', 'start_node_lat', 'start_loc_id',
       'start_loc_name', 'end_node_lon', 'end_node_lat', 'end_loc_id',
       'end_loc_name'],
      dtype='str')

In [18]:
# save the stations info = duration might be used later for graph
"""bike_trips_locations[[
        "end_station_name",
        "end_loc_id",
        "start_station_name",
        "start_loc_id",
        "avg_trip_duration",
    ]
].sort_values(
    "end_station_name"
).drop_duplicates().to_parquet(project_root / "data/raw/graph/stations_flow_duration_5yrs_12pm.parquet", index=False)"""


'bike_trips_locations[[\n        "end_station_name",\n        "end_loc_id",\n        "start_station_name",\n        "start_loc_id",\n        "avg_trip_duration",\n    ]\n].sort_values(\n    "end_station_name"\n).drop_duplicates().to_parquet(project_root / "data/raw/graph/stations_flow_duration_5yrs_12pm.parquet", index=False)'

In [19]:
# write steps for the rest how inflo matrix / outflow matrix

In [20]:
# to build the base table:
# X 1. calculate the spatial dispersion per each end station add number of origin_stations total_end_capacities and avg_total_capacities
# X 2. Calculate the duration disterbution per each end station
# 3. left join with node on loc_id to get all associated info => static features

# 4. transfer the weather data into categorical
# 5  create season pd as start_date, end_date, season
# 6. aggr = end_node_loc_id, end_stations_names, sum count_of_trips 
# merge 5,6 and holiday excel with 2 


In [21]:
# 2. inflow matrix

inflow = (
    bike_trips_locations[
        ["end_date","end_station_name", "end_loc_id", "capacity_end", "trip_count"]
    ]
    .dropna(subset=["end_date","end_station_name", "end_loc_id", "trip_count"])
    .groupby(["end_date","end_station_name", "end_loc_id"], as_index=False)
    .agg(
        end_capacity_avg=("capacity_end", "mean"),
        end_trips_count=("trip_count", "sum"),
    ) 
    .rename(columns={"end_date": "date"})
)

outflow = (
    bike_trips_locations[
        ["end_date", "start_station_name", "start_loc_id", "capacity_start", "trip_count"]
    ]
    .dropna(subset=["end_date", "start_station_name", "start_loc_id", "trip_count"])
    .groupby(["end_date", "start_station_name", "start_loc_id"], as_index=False)
    .agg(
        start_capacity_avg=("capacity_start", "mean"),
        start_trips_count=("trip_count", "sum"),
    )
    .rename(columns={"end_date": "date"})
)

inflow.head(), outflow.head()

(        date                             end_station_name  end_loc_id  \
 0 2022-01-01  111 Bond St (North of Dundas St E)  - SMART  5350034.02   
 1 2022-01-01                        1303 Yonge St - SMART  5350124.00   
 2 2022-01-01         161 Bleecker St (South of Wellesley)  5350066.00   
 3 2022-01-01                                 25 Booth Ave  5350001.00   
 4 2022-01-01             25 York St – Union Station South  5350013.01   
 
    end_capacity_avg  end_trips_count  
 0              16.0                2  
 1              15.0                1  
 2              23.0                1  
 3              24.0                2  
 4              11.0                1  ,
         date                    start_station_name start_loc_id  \
 0 2022-01-01                         12 Harbour St   5350013.02   
 1 2022-01-01                 1303 Yonge St - SMART   5350124.00   
 2 2022-01-01  161 Bleecker St (South of Wellesley)   5350066.00   
 3 2022-01-01                     190 Que

In [22]:
#outflow #692,609
outflow #666,381

,date,start_station_name,start_loc_id,start_capacity_avg,start_trips_count
0,2022-01-01,12 Harbour St,5350013.02,15.0,2
1,2022-01-01,1303 Yonge St - SMART,5350124.00,15.0,1
2,2022-01-01,161 Bleecker St (South of Wellesley),5350066.00,23.0,1
3,2022-01-01,190 Queens Quay E,5350017.02,15.0,2
4,2022-01-01,25 Booth Ave,5350001.00,24.0,1
...,...,...,...,...,...
692604,2026-03-31,York St / Queen St W (City Hall),5350035.00,26.0,1
692605,2026-03-31,York St / Queens Quay W,5350012.04,57.0,13
692606,2026-03-31,York St / Wellington St W (1),5350014.00,22.0,1
692607,2026-03-31,York University (Glendon Campus) - SMART,5350265.00,16.0,1


In [23]:
# 3
nodes = pd.read_parquet(
    "/home/najla/dev/najla-msc/data/processed/THATS/NetworkGraph/nodes.parquet"
).drop(columns=['geometry',"type","original_geometry"]).reset_index()
nodes.shape

(1248, 42)

In [24]:
nodes.head()

,loc_id,loc_name,lon,lat,population,jobs,area,poi_arts_and_entertainment,poi_community_and_government,poi_cultural_and_historic,...,land_use_military,land_use_park,land_use_pedestrian,land_use_protected,land_use_recreation,land_use_religious,land_use_residential,land_use_resource_extraction,land_use_transportation,land_use_winter_sports
0,5320100.01,0100.01,-78.935020,43.856672,0.000451,0.000203,16220000.0,4.637299e-07,1.391190e-06,6.376286e-07,...,0.0,0.003531,0.000044,0.003211,0.000218,0.000000,0.012452,0.000000,0.001572,0.0
1,5320100.02,0100.02,-78.966386,43.869987,0.002052,0.000875,2680000.0,3.708509e-07,3.708509e-07,3.708509e-07,...,0.0,0.010381,0.000000,0.000000,0.000587,0.000000,0.134664,0.000000,0.000000,0.0
2,5320100.03,0100.03,-78.972803,43.882699,0.001365,0.000666,5940000.0,3.367461e-07,3.367461e-07,6.734923e-07,...,0.0,0.014758,0.000000,0.000000,0.000423,0.002103,0.041123,0.000000,0.000000,0.0
3,5320105.14,0105.14,-78.985482,43.927658,0.000334,0.000157,15920000.0,1.250645e-07,3.126613e-07,1.250645e-07,...,0.0,0.003706,0.000066,0.000000,0.000069,0.000000,0.017175,0.024266,0.000000,0.0
4,5320105.17,0105.17,-78.992870,43.987493,0.000032,0.000016,52030000.0,3.834821e-08,1.342188e-07,1.150446e-07,...,0.0,0.000096,0.000000,0.000000,0.000019,0.001239,0.000000,0.000000,0.000000,0.0


In [25]:
# 1. Stack start and end stations
stacked_stations = pd.concat(
    [
        bike_trips_locations[
            ["start_station_id", "start_loc_id", "capacity_start"]
        ].rename(
            columns={
                "start_station_id": "station_id",
                "start_loc_id": "loc_id",
                "capacity_start": "capacity",
            }
        ),
        bike_trips_locations[
            ["end_station_id", "end_loc_id", "capacity_end"]
        ].rename(
            columns={
                "end_station_id": "station_id",
                "end_loc_id": "loc_id",
                "capacity_end": "capacity",
            }
        ),
    ],
    ignore_index=True,
).dropna(subset=["station_id", "loc_id"])

# avg capacity per station, then sum on node level
station_capacity_features = (
    stacked_stations
    .groupby(["loc_id", "station_id"], as_index=False)
    .agg(station_capacity_avg=("capacity", "mean"))
    .groupby("loc_id", as_index=False)
    .agg(stations_capacity_sum=("station_capacity_avg", "sum"))
)

# 3. count start stations per node
start_station_counts = (
    bike_trips_locations
    .dropna(subset=["start_loc_id", "start_station_id"])
    .groupby("start_loc_id", as_index=False)
    .agg(start_stations_count=("start_station_id", "nunique"))
)

# 4. count end stations per node
end_station_counts = (
    bike_trips_locations
    .dropna(subset=["end_loc_id", "end_station_id"])
    .groupby("end_loc_id", as_index=False)
    .agg(end_stations_count=("end_station_id", "nunique"))
)

In [26]:
# 5. Join only with nodes
static_features = (
    nodes.merge(
        station_capacity_features,
        on="loc_id",
        how="left",
    )
    .merge(
        start_station_counts,
        left_on="loc_id",
        right_on="start_loc_id",
        how="left",
    )
    .merge(
        end_station_counts,
        left_on="loc_id",
        right_on="end_loc_id",
        how="left",
    )
    .drop(
        columns=["start_loc_id", "end_loc_id"],
        errors="ignore",
    )
)


static_features["total_stations"] = (
    static_features["start_stations_count"]
    + static_features["end_stations_count"]
)

static_features

,loc_id,loc_name,lon,lat,population,jobs,area,poi_arts_and_entertainment,poi_community_and_government,poi_cultural_and_historic,...,land_use_recreation,land_use_religious,land_use_residential,land_use_resource_extraction,land_use_transportation,land_use_winter_sports,stations_capacity_sum,start_stations_count,end_stations_count,total_stations
0,5320100.01,0100.01,-78.935020,43.856672,0.000451,0.000203,16220000.0,4.637299e-07,1.391190e-06,6.376286e-07,...,0.000218,0.000000,0.012452,0.000000,0.001572,0.0,NaN,NaN,NaN,NaN
1,5320100.02,0100.02,-78.966386,43.869987,0.002052,0.000875,2680000.0,3.708509e-07,3.708509e-07,3.708509e-07,...,0.000587,0.000000,0.134664,0.000000,0.000000,0.0,NaN,NaN,NaN,NaN
2,5320100.03,0100.03,-78.972803,43.882699,0.001365,0.000666,5940000.0,3.367461e-07,3.367461e-07,6.734923e-07,...,0.000423,0.002103,0.041123,0.000000,0.000000,0.0,NaN,NaN,NaN,NaN
3,5320105.14,0105.14,-78.985482,43.927658,0.000334,0.000157,15920000.0,1.250645e-07,3.126613e-07,1.250645e-07,...,0.000069,0.000000,0.017175,0.024266,0.000000,0.0,NaN,NaN,NaN,NaN
4,5320105.17,0105.17,-78.992870,43.987493,0.000032,0.000016,52030000.0,3.834821e-08,1.342188e-07,1.150446e-07,...,0.000019,0.001239,0.000000,0.000000,0.000000,0.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1243,5680101.00,0101.00,-79.639200,44.242834,0.000033,0.000016,170910000.0,4.091783e-08,8.183566e-08,6.429944e-08,...,0.000040,0.000013,0.000685,0.000000,0.000000,0.0,NaN,NaN,NaN,NaN
1244,5680102.01,0102.01,-79.562378,44.386827,0.000161,0.000076,19120000.0,2.431003e-08,4.862005e-08,2.431003e-08,...,0.000167,0.000000,0.005397,0.000000,0.000000,0.0,NaN,NaN,NaN,NaN
1245,5680103.05,0103.05,-79.528533,44.331678,0.001307,0.000642,3170000.0,3.493472e-07,2.328981e-07,0.000000e+00,...,0.001347,0.000000,0.066406,0.000000,0.000000,0.0,NaN,NaN,NaN,NaN
1246,5680103.08,0103.08,-79.528152,44.308072,0.001540,0.000703,2410000.0,0.000000e+00,2.554194e-07,0.000000e+00,...,0.000315,0.000000,0.132143,0.000000,0.000000,0.0,NaN,NaN,NaN,NaN


In [27]:
# number of not null
static_features[["stations_capacity_sum","start_stations_count","end_stations_count","total_stations",]].notna().sum()

stations_capacity_sum    364
start_stations_count     363
end_stations_count       364
total_stations           363
dtype: int64

In [28]:
# 4

# =========================
# 1. File path
# =========================

# Read weather data
weather = pd.read_csv(project_root / "data/raw/bike_ridership/temporal/toronto_weather_2022_2026.csv")

# Convert Date
weather["Date"] = pd.to_datetime(weather["Date"])

# =========================
# 2. Thresholds
# =========================
SNOW_CM_THRESHOLD = 0.0
RAIN_MM_THRESHOLD = 5.0
WIND_KMH_THRESHOLD = 30.0
VERY_COLD_MEAN_TEMP_C = 0.0
VERY_HOT_MAX_TEMP_C = 30.0

# =========================
# 3. Weather category
# Priority:
# Snowy > Rainy > Windy > Very cold > Very hot > Clear weather
# =========================
conditions = [
    weather["Snow_cm"].fillna(0) > SNOW_CM_THRESHOLD,
    weather["Rain_mm"].fillna(0) >= RAIN_MM_THRESHOLD,
    weather["Max_Wind_Speed_kmh"].fillna(0) >= WIND_KMH_THRESHOLD,
    weather["Mean_Temp_C"] <= VERY_COLD_MEAN_TEMP_C,
    weather["Max_Temp_C"] >= VERY_HOT_MAX_TEMP_C
]

choices = [
    "snowy",
    "rainy",
    "windy",
    "very_cold",
    "very_hot"
]

weather["weather_category"] = np.select(
    conditions,
    choices,
    default="clear_weather"
)

# =========================
# 4. Final dataframe
# =========================
weather_final = weather[["Date", "weather_category"]]

weather_final

,Date,weather_category
0,2022-01-01,snowy
1,2022-01-02,snowy
2,2022-01-03,very_cold
3,2022-01-04,very_cold
4,2022-01-05,windy
...,...,...
1547,2026-03-28,very_cold
1548,2026-03-29,clear_weather
1549,2026-03-30,clear_weather
1550,2026-03-31,rainy


In [29]:
weather_final.groupby("weather_category")["Date"].count()

weather_category
clear_weather    873
rainy            175
snowy            293
very_cold        135
very_hot          25
windy             51
Name: Date, dtype: int64

In [30]:
# 5.season
import pandas as pd

season = pd.DataFrame({
    "start_date": [
        "2022-01-01",
        "2022-03-01",
        "2022-06-01",
        "2022-09-01",
        "2022-12-01",
        "2023-03-01",
        "2023-06-01",
        "2023-09-01",
        "2023-12-01",
        "2024-03-01",
        "2024-06-01",
        "2024-09-01",
        "2024-12-01",
        "2025-03-01",
        "2025-06-01",
        "2025-09-01",
        "2025-12-01",
        "2026-03-01",
        "2026-06-01",
        "2026-09-01",
        "2026-12-01",
    ],
    "end_date": [
        "2022-02-28",
        "2022-05-31",
        "2022-08-31",
        "2022-11-30",
        "2023-02-28",
        "2023-05-31",
        "2023-08-31",
        "2023-11-30",
        "2024-02-29",
        "2024-05-31",
        "2024-08-31",
        "2024-11-30",
        "2025-02-28",
        "2025-05-31",
        "2025-08-31",
        "2025-11-30",
        "2026-02-28",
        "2026-05-31",
        "2026-08-31",
        "2026-11-30",
        "2027-02-28",
    ],
    "season": [
        "winter",
        "spring",
        "summer",
        "autumn",
        "winter",
        "spring",
        "summer",
        "autumn",
        "winter",
        "spring",
        "summer",
        "autumn",
        "winter",
        "spring",
        "summer",
        "autumn",
        "winter",
        "spring",
        "summer",
        "autumn",
        "winter",
    ]
})

season["start_date"] = pd.to_datetime(season["start_date"])
season["end_date"] = pd.to_datetime(season["end_date"])

season

,start_date,end_date,season
0,2022-01-01,2022-02-28,winter
1,2022-03-01,2022-05-31,spring
2,2022-06-01,2022-08-31,summer
3,2022-09-01,2022-11-30,autumn
4,2022-12-01,2023-02-28,winter
5,2023-03-01,2023-05-31,spring
6,2023-06-01,2023-08-31,summer
7,2023-09-01,2023-11-30,autumn
8,2023-12-01,2024-02-29,winter
9,2024-03-01,2024-05-31,spring


In [31]:
# 5.Holiday data
holidays_weekends = pd.read_csv(project_root / "data/raw/bike_ridership/temporal/toronto_holidays_weekends_2022_2026.csv")
holidays_weekends

,date,holiday_type
0,2022-01-01,"New Year's Day, Weekend"
1,2022-01-02,Weekend
2,2022-01-08,Weekend
3,2022-01-09,Weekend
4,2022-01-15,Weekend
...,...,...
484,2026-03-15,Weekend
485,2026-03-21,Weekend
486,2026-03-22,Weekend
487,2026-03-28,Weekend


In [32]:
# 6. static features
# --------------------------------
# 1. aggregate bike trips by [day, end node]
# --------------------------------
bike_daily = inflow[["date"]].drop_duplicates()
bike_daily

,date
0,2022-01-01
207,2022-01-02
282,2022-01-03
406,2022-01-04
584,2022-01-05
...,...
663651,2026-03-27
664176,2026-03-28
664710,2026-03-29
665285,2026-03-30


In [33]:
# --------------------------------
# 2. left join holidays_weekends
# --------------------------------
bike_daily["date"] = pd.to_datetime(bike_daily["date"]).dt.normalize()
holidays_weekends["date"] = pd.to_datetime(holidays_weekends["date"]).dt.normalize()

dynamic_features = bike_daily.merge(
    holidays_weekends,
    left_on="date",
    right_on="date",
    how="left",
)

dynamic_features["day_type"] = dynamic_features["holiday_type"].apply(
    lambda x: "Weekend" if pd.notna(x) and "Weekend" in x else "weekday"
)

dynamic_features["public_holiday"] = dynamic_features["holiday_type"].apply(
    lambda x: (
        "no_public_holiday"
        if pd.isna(x) or x == "weekday"
        else ", ".join(
            [part.strip() for part in x.split(",") if part.strip() != "Weekend"]
        ) or "no_public_holiday"
    )
)
dynamic_features = dynamic_features.drop(columns=["holiday_type"], errors="ignore")

# --------------------------------
# 3. left join weather
# --------------------------------
dynamic_features = dynamic_features.merge(
    weather_final,
    left_on="date",
    right_on="Date",
    how="left",
)


dynamic_features = dynamic_features.drop(columns=["Date"], errors="ignore")


dynamic_features

,date,day_type,public_holiday,weather_category
0,2022-01-01,Weekend,New Year's Day,snowy
1,2022-01-02,Weekend,no_public_holiday,snowy
2,2022-01-03,weekday,no_public_holiday,very_cold
3,2022-01-04,weekday,no_public_holiday,very_cold
4,2022-01-05,weekday,no_public_holiday,windy
...,...,...,...,...
1546,2026-03-27,weekday,no_public_holiday,very_cold
1547,2026-03-28,Weekend,no_public_holiday,very_cold
1548,2026-03-29,Weekend,no_public_holiday,clear_weather
1549,2026-03-30,weekday,no_public_holiday,clear_weather


In [34]:
season = season.rename(
    columns={
        "start_date": "season_start_date",
        "end_date": "season_end_date",
    }
)

season["season_start_date"] = pd.to_datetime(season["season_start_date"])
season["season_end_date"] = pd.to_datetime(season["season_end_date"])

season

,season_start_date,season_end_date,season
0,2022-01-01,2022-02-28,winter
1,2022-03-01,2022-05-31,spring
2,2022-06-01,2022-08-31,summer
3,2022-09-01,2022-11-30,autumn
4,2022-12-01,2023-02-28,winter
5,2023-03-01,2023-05-31,spring
6,2023-06-01,2023-08-31,summer
7,2023-09-01,2023-11-30,autumn
8,2023-12-01,2024-02-29,winter
9,2024-03-01,2024-05-31,spring


In [35]:
dynamic_features["day_type"] = (
    dynamic_features["day_type"]
    .astype(str)
    .str.lower()
)

dynamic_features["public_holiday"] = (
    dynamic_features["public_holiday"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(" ", "_", regex=False)
    .str.replace("'", "", regex=False)
)

dynamic_features["season"] = pd.NA

for _, row in season.iterrows():
    mask = dynamic_features["date"].between(row["season_start_date"], row["season_end_date"])
    dynamic_features.loc[mask, "season"] = row["season"]

dynamic_features

,date,day_type,public_holiday,weather_category,season
0,2022-01-01,weekend,new_years_day,snowy,winter
1,2022-01-02,weekend,no_public_holiday,snowy,winter
2,2022-01-03,weekday,no_public_holiday,very_cold,winter
3,2022-01-04,weekday,no_public_holiday,very_cold,winter
4,2022-01-05,weekday,no_public_holiday,windy,winter
...,...,...,...,...,...
1546,2026-03-27,weekday,no_public_holiday,very_cold,spring
1547,2026-03-28,weekend,no_public_holiday,very_cold,spring
1548,2026-03-29,weekend,no_public_holiday,clear_weather,spring
1549,2026-03-30,weekday,no_public_holiday,clear_weather,spring


In [36]:
# 1. read the text_data and drop duplicates on (label, instance and geom)
# 2. intersects with loc_id
# 3. convert into label, set[instanct], set[loc_id].. check how many loc_id?

In [37]:
visitors_points = (
    gpd.read_file(
        "/home/najla/dev/najla-msc/bikeshare/data/raw/bike_ridership/attractions/all_types_visitors_points.gpkg"
    )[["itemLabel", "instanceOfLabel", "geometry"]]
    .drop_duplicates()
    .sort_values(by=["itemLabel", "instanceOfLabel"])
    .reset_index(drop=True))
visitors_points

,itemLabel,instanceOfLabel,geometry
0,1 Spadina Crescent,university building,POINT (-79.4008 43.6597)
1,10 Dundas East,building complex,POINT (-79.38073 43.65676)
2,10 Dundas East,entertainment venue,POINT (-79.38073 43.65676)
3,10 Dundas East,shopping center,POINT (-79.38073 43.65676)
4,10 Navy,skyscraper,POINT (-79.3919 43.6404)
...,...,...,...
3252,École secondaire Toronto Ouest,high school,POINT (-79.441 43.6522)
3253,École secondaire catholique Saint-Frère-André,high school,POINT (-79.441 43.6522)
3254,Église du Sacré-Coeur,church building,POINT (-79.37285 43.66359)
3255,ÏCE Condominiums at York Centre,building complex,POINT (-79.38158 43.64206)


In [38]:
visitors_points.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [39]:
# 1. Project data first
visitors_proj = visitors_points.to_crs("EPSG:3978")

# 2. Get central point per itemLabel using geometric median
rows = []

for item, group in visitors_proj.groupby("itemLabel"):
    
    coords = np.array([[p.x, p.y] for p in group.geometry])
    
    start = coords.mean(axis=0)

    result = minimize(
        lambda x: np.sqrt(((coords - x) ** 2).sum(axis=1)).sum(),
        start
    )

    rows.append({
        "itemLabel": item,
        "instanceOfLabel": ", ".join(sorted(set(group["instanceOfLabel"].dropna()))),
        "geometry": Point(result.x[0], result.x[1])
    })

# 3. Convert back to GeoDataFrame
visitors_points_avg = gpd.GeoDataFrame(
    rows,
    geometry="geometry",
    crs="EPSG:3978"
).to_crs("EPSG:4326")

# 4. Spatial join
wikidata_loc = (
    gpd.sjoin(
        visitors_points_avg,
        locations_gdf[["loc_id", "loc_name", "geometry"]],
        how="left",
        predicate="within",
    )
    .drop(columns=["index_right","geometry","loc_name"], errors="ignore")
)
wikidata_loc.sort_values(by=["itemLabel", "instanceOfLabel"]).reset_index(drop=True)

,itemLabel,instanceOfLabel,loc_id
0,1 Spadina Crescent,university building,5350061.00
1,10 Dundas East,"building complex, entertainment venue, shoppin...",5350034.02
2,10 Navy,skyscraper,5350012.03
3,"121 St. George Street, Toronto",university building,5350061.00
4,"123 Edward Street, Toronto",university building,5350035.00
...,...,...,...
2246,statue of Alexander Wood,"LGBT monument or memorial, statue",5350063.06
2247,École secondaire Toronto Ouest,high school,5350053.00
2248,École secondaire catholique Saint-Frère-André,high school,5350053.00
2249,Église du Sacré-Coeur,church building,5350066.00


In [40]:
wikidata_loc["loc_id"].nunique()

469

In [41]:
#Test
visitors_points_with_count=wikidata_loc.groupby("itemLabel", as_index=False).agg(loc_id_count=("loc_id", "nunique")).sort_values("loc_id_count", ascending=False)
visitors_points_with_count["loc_id_count"] > 1

2250    False
0       False
1       False
2       False
3       False
        ...  
1147    False
917     False
919     False
477     False
70      False
Name: loc_id_count, Length: 2251, dtype: bool

In [42]:
#Test
visitors_points_with_count[visitors_points_with_count["loc_id_count"] > 1].count()

itemLabel       0
loc_id_count    0
dtype: int64

In [43]:
error

NameError: name 'error' is not defined

In [ ]:
#Test all 
dynamic_features[["end_date","end_loc_id"]].merge(static_features, left_on="end_loc_id", right_on="loc_id", how="right") \
.merge(wikidata_loc, left_on="end_loc_id", right_on="loc_id", how="left")

KeyError: "None of [Index(['end_date', 'end_loc_id'], dtype='str')] are in the [columns]"

In [ ]:
error

In [ ]:
# save files
save_path = Path("/home/najla/dev/najla-msc/bikeshare/data/processed/bike_ridership")

In [ ]:

dynamic_features.to_parquet(save_path / "dynamic_features_5yrs_12pm.parquet", index=False)

In [ ]:
static_features.to_parquet(save_path / "static_features_5yrs_12pm.parquet", index=False)

In [ ]:
wikidata_loc.to_parquet(save_path / "wikidata_loc_5yrs_12pm.parquet", index=False)

In [ ]:
inflow.to_parquet(save_path / "inflow_5yrs_12pm.parquet", index=False)

In [ ]:
outflow.to_parquet(save_path / "outflow_5yrs_12pm.parquet", index=False)

In [ ]:
dynamic_features.columns

Index(['end_date', 'end_loc_id', 'end_loc_name', 'trip_count', 'holiday_type',
       'Main_Weather_Category', 'season'],
      dtype='str')

In [ ]:
static_features.columns

Index(['loc_id', 'loc_name', 'population', 'jobs', 'area',
       'poi_arts_and_entertainment', 'poi_community_and_government',
       'poi_cultural_and_historic', 'poi_education', 'poi_food_and_drink',
       'poi_geographic_entities', 'poi_health_care', 'poi_lifestyle_services',
       'poi_lodging', 'poi_services_and_business', 'poi_shopping',
       'poi_sports_and_recreation', 'poi_travel_and_transportation',
       'land_use_agriculture', 'land_use_campground', 'land_use_cemetery',
       'land_use_construction', 'land_use_developed', 'land_use_education',
       'land_use_entertainment', 'land_use_golf', 'land_use_horticulture',
       'land_use_landfill', 'land_use_managed', 'land_use_medical',
       'land_use_military', 'land_use_park', 'land_use_pedestrian',
       'land_use_protected', 'land_use_recreation', 'land_use_religious',
       'land_use_residential', 'land_use_resource_extraction',
       'land_use_transportation', 'land_use_winter_sports',
       'spatial_dispers

In [ ]:
wikidata_loc.columns

Index(['itemLabel', 'instanceOfLabel', 'loc_id'], dtype='str')

Explore the years

In [ ]:
from datetime import date

bike_trips.filter(
    pl.col("end_date") > date(2025, 1, 1)
)

end_date,end_station_name,start_station_name,avg_trip_duration,trip_count,end_station_id,lat_end,lon_end,capacity_end,start_station_id,lat_start,lon_start,capacity_start
date,str,str,f64,u32,i64,f64,f64,i64,i64,f64,f64,i64
